# Electric Vehicle Population Analysis — Data Cleaning

This notebook prepares the Washington State electric vehicle population dataset for analysis.

I am using this dataset because EV adoption is a current topic, and the data is public, practical, and easy to connect with real business questions.

Important note: this dataset shows registered electric vehicles in Washington State. It should not be interpreted as global EV sales data.

In this notebook, I will:

- load the raw CSV data
- clean column names
- check missing values
- remove columns that are not needed for this analysis
- create a cleaned CSV file for the next notebooks

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load the dataset

The data comes from Washington State's public EV population dataset.

I keep the raw file locally in `data/raw/`. If the file is not already there, this cell downloads it from the public data portal.

In [2]:
raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

raw_path = raw_dir / "electric_vehicle_population.csv"

url = "https://data.wa.gov/api/views/f6w7-q2d2/rows.csv?accessType=DOWNLOAD"

if raw_path.exists():
    print("Loading local file...")
    ev_df = pd.read_csv(raw_path, low_memory=False)
else:
    print("Downloading dataset...")
    ev_df = pd.read_csv(url, low_memory=False)
    ev_df.to_csv(raw_path, index=False)
    print("Saved raw file to:", raw_path)

ev_df.head()

Saved raw file to: ..\data\raw\electric_vehicle_population.csv


,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract
0,WBY2Z2C57G,Yakima,Yakima,WA,98908.0,2016,BMW,I8,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,14.0,14.0,232628763,POINT (-120.60199 46.59817),PACIFICORP,5.307700e+10
1,5YJYGDEE8M,Yakima,Selah,WA,98942.0,2021,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,15.0,236581355,POINT (-120.53145 46.65405),PACIFICORP,5.307700e+10
2,JTDKN3DP9D,Snohomish,Marysville,WA,98271.0,2013,TOYOTA,PRIUS,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,6.0,38.0,211956958,POINT (-122.17138 48.10433),PUGET SOUND ENERGY INC,5.306105e+10
3,WP1AE2AY6M,Yakima,Yakima,WA,98908.0,2021,PORSCHE,CAYENNE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,16.0,14.0,271976883,POINT (-120.60199 46.59817),PACIFICORP,5.307700e+10
4,LPSED3KA5M,Kitsap,Bainbridge Island,WA,98110.0,2021,POLESTAR,PS2,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,233.0,23.0,255446103,POINT (-122.521 47.62759),PUGET SOUND ENERGY INC,5.303509e+10


## 2. First look

Before cleaning, I check the size of the dataset and the column names.

In [3]:
print("Rows:", ev_df.shape[0])
print("Columns:", ev_df.shape[1])

Rows: 285822
Columns: 16


In [4]:
ev_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285822 entries, 0 to 285821
Data columns (total 16 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   VIN (1-10)                                         285822 non-null  object 
 1   County                                             285812 non-null  object 
 2   City                                               285812 non-null  object 
 3   State                                              285822 non-null  object 
 4   Postal Code                                        285812 non-null  float64
 5   Model Year                                         285822 non-null  int64  
 6   Make                                               285822 non-null  object 
 7   Model                                              285822 non-null  object 
 8   Electric Vehicle Type                              285822 non-null  object

In [5]:
ev_df.columns

Index(['VIN (1-10)', 'County', 'City', 'State', 'Postal Code', 'Model Year',
       'Make', 'Model', 'Electric Vehicle Type',
       'Clean Alternative Fuel Vehicle (CAFV) Eligibility', 'Electric Range',
       'Legislative District', 'DOL Vehicle ID', 'Vehicle Location',
       'Electric Utility', '2020 Census Tract'],
      dtype='object')

The original column names are readable, but not very convenient for Python.

I will convert them into lowercase names with underscores.

In [6]:
ev_df.columns = (
    ev_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("/", "_")
)

ev_df.columns

Index(['vin_1_10', 'county', 'city', 'state', 'postal_code', 'model_year',
       'make', 'model', 'electric_vehicle_type',
       'clean_alternative_fuel_vehicle_cafv_eligibility', 'electric_range',
       'legislative_district', 'dol_vehicle_id', 'vehicle_location',
       'electric_utility', '2020_census_tract'],
      dtype='object')

## 3. Missing values

I check missing values before deciding what to keep or remove.

Some missing values are expected, especially for location-related fields or electric range.

In [7]:
missing = ev_df.isna().sum().sort_values(ascending=False)

missing_table = pd.DataFrame({
    "column": missing.index,
    "missing_values": missing.values,
    "missing_percent": (missing.values / len(ev_df) * 100).round(2)
})

missing_table

,column,missing_values,missing_percent
0,legislative_district,733,0.26
1,vehicle_location,18,0.01
2,county,10,0.00
3,city,10,0.00
4,2020_census_tract,10,0.00
5,electric_utility,10,0.00
6,postal_code,10,0.00
7,electric_range,8,0.00
8,model,0,0.00
9,make,0,0.00


## 4. Check duplicates

Duplicate rows can affect counts by brand, county, and model year.

I check for exact duplicate rows.

In [9]:
duplicate_count = ev_df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [10]:
rows_before = len(ev_df)

ev_df = ev_df.drop_duplicates()

rows_after = len(ev_df)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)

Rows before: 285822
Rows after: 285822
Rows removed: 0


## 5. Keep useful columns

For this project, I only need columns that help with EV population analysis.

I keep fields related to location, model year, make, model, EV type, electric range, and utility area.

In [11]:
ev_df.columns

Index(['vin_1_10', 'county', 'city', 'state', 'postal_code', 'model_year',
       'make', 'model', 'electric_vehicle_type',
       'clean_alternative_fuel_vehicle_cafv_eligibility', 'electric_range',
       'legislative_district', 'dol_vehicle_id', 'vehicle_location',
       'electric_utility', '2020_census_tract'],
      dtype='object')

In [13]:
columns_to_keep = [
    "county",
    "city",
    "state",
    "postal_code",
    "model_year",
    "make",
    "model",
    "electric_vehicle_type",
    "clean_alternative_fuel_vehicle_cafv_eligibility",
    "electric_range",
    "legislative_district",
    "electric_utility"
]

ev_clean = ev_df[columns_to_keep].copy()

ev_clean.head()

,county,city,state,postal_code,model_year,make,model,electric_vehicle_type,clean_alternative_fuel_vehicle_cafv_eligibility,electric_range,legislative_district,electric_utility
0,Yakima,Yakima,WA,98908.0,2016,BMW,I8,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,14.0,14.0,PACIFICORP
1,Yakima,Selah,WA,98942.0,2021,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,15.0,PACIFICORP
2,Snohomish,Marysville,WA,98271.0,2013,TOYOTA,PRIUS,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,6.0,38.0,PUGET SOUND ENERGY INC
3,Yakima,Yakima,WA,98908.0,2021,PORSCHE,CAYENNE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,16.0,14.0,PACIFICORP
4,Kitsap,Bainbridge Island,WA,98110.0,2021,POLESTAR,PS2,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,233.0,23.0,PUGET SOUND ENERGY INC


## 6. Clean text columns

I clean text columns so values like brand names, cities, and EV types are more consistent.

In [15]:
text_columns = ev_clean.select_dtypes(include="object").columns

for col in text_columns:
    ev_clean[col] = (
        ev_clean[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

ev_clean.head()

,county,city,state,postal_code,model_year,make,model,electric_vehicle_type,clean_alternative_fuel_vehicle_cafv_eligibility,electric_range,legislative_district,electric_utility
0,Yakima,Yakima,WA,98908.0,2016,BMW,I8,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,14.0,14.0,PACIFICORP
1,Yakima,Selah,WA,98942.0,2021,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,15.0,PACIFICORP
2,Snohomish,Marysville,WA,98271.0,2013,TOYOTA,PRIUS,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,6.0,38.0,PUGET SOUND ENERGY INC
3,Yakima,Yakima,WA,98908.0,2021,PORSCHE,CAYENNE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,16.0,14.0,PACIFICORP
4,Kitsap,Bainbridge Island,WA,98110.0,2021,POLESTAR,PS2,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,233.0,23.0,PUGET SOUND ENERGY INC


In [16]:
ev_clean[["model_year", "make", "model", "electric_vehicle_type", "electric_range"]].head()

,model_year,make,model,electric_vehicle_type,electric_range
0,2016,BMW,I8,Plug-in Hybrid Electric Vehicle (PHEV),14.0
1,2021,TESLA,MODEL Y,Battery Electric Vehicle (BEV),0.0
2,2013,TOYOTA,PRIUS,Plug-in Hybrid Electric Vehicle (PHEV),6.0
3,2021,PORSCHE,CAYENNE,Plug-in Hybrid Electric Vehicle (PHEV),16.0
4,2021,POLESTAR,PS2,Battery Electric Vehicle (BEV),233.0


In [17]:
ev_clean["electric_vehicle_type"].value_counts()

electric_vehicle_type
Battery Electric Vehicle (BEV)            229876
Plug-in Hybrid Electric Vehicle (PHEV)     55946
Name: count, dtype: int64

In [18]:
ev_clean["make"].value_counts().head(10)

make
TESLA         117392
CHEVROLET      19677
NISSAN         16121
FORD           15639
KIA            14328
TOYOTA         12234
BMW            11688
HYUNDAI        11067
RIVIAN          9098
VOLKSWAGEN      7658
Name: count, dtype: int64

In [19]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

clean_path = processed_dir / "ev_population_cleaned.csv"

ev_clean.to_csv(clean_path, index=False)

print("Saved cleaned data to:", clean_path)

Saved cleaned data to: ..\data\processed\ev_population_cleaned.csv
